In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# FairRecourse: Computational Audit Report\n",
    "This notebook provides the end-to-end reproducible experiment comparing baseline predictive performance, fairness auditing, and constrained counterfactual recourse."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.data_loader import load_and_preprocess_german_credit\n",
    "from src.model import CalibratedRiskModel\n",
    "from src.explain import ExplainerEngine\n",
    "from src.recourse import ConstrainedRecourseGenerator\n",
    "from src.policy_layer import PolicyAndConstraintLayer\n",
    "from src.cost_feasibility import RecourseEvaluator\n",
    "from src.audit import FairnessAuditor\n",
    "from src.certificate import generate_recourse_certificate\n",
    "\n",
    "X_train, X_test, y_train, y_test, A_train, A_test = load_and_preprocess_german_credit()\n",
    "model = CalibratedRiskModel().fit(X_train, y_train)\n",
    "print(\"Model Metrics:\", model.evaluate(X_test, y_test))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate individual case recourse\n",
    "instance = X_test.iloc[[1]]\n",
    "prob = model.predict_proba(instance)[0]\n",
    "explainer = ExplainerEngine(model, X_train)\n",
    "drivers = explainer.explain_instance(instance, top_k=3).to_dict(orient=\"records\")\n",
    "\n",
    "generator = ConstrainedRecourseGenerator(model.calibrated_model, X_train, continuous_features=list(X_train.columns))\n",
    "cfs = generator.generate(instance, total_cfs=1)\n",
    "target_cf = cfs.iloc[0].drop(labels=[\"target\"], errors=\"ignore\")\n",
    "\n",
    "pcl = PolicyAndConstraintLayer()\n",
    "is_valid, violations = pcl.validate(instance.iloc[0], target_cf)\n",
    "eval_res = RecourseEvaluator().evaluate(instance.iloc[0], target_cf, violations)\n",
    "\n",
    "print(generate_recourse_certificate(\n",
    "    audit_id=\"NB-AUDIT-001\", applicant_id=\"APP-001\",\n",
    "    baseline_pd=prob, threshold=0.40, shap_drivers=drivers,\n",
    "    baseline_profile=instance.iloc[0].to_dict(), recourse_plan=target_cf.to_dict(),\n",
    "    eval_result=eval_res\n",
    "))"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}